# Dataclasses – Practice Exercises

This notebook contains hands-on exercises to practice Python's `dataclasses`.

The exercises follow the same progression as the guide: core ideas, fields/defaults, `__post_init__`, immutability/comparison, inheritance, and finally a trading-flavored case study.


### Contents

- [Exercise 1 – Minimal Dataclass](#exercise-1--minimal-dataclass)
- [Exercise 2 – Defaults and `field()`](#exercise-2--defaults-and-field)
- [Exercise 3 – `__post_init__` Validation](#exercise-3--post_init-validation)
- [Exercise 4 – `frozen=True` Immutability](#exercise-4--frozentrue-immutability)
- [Exercise 5 – `slots=True` and Inheritance](#exercise-5--slotstrue-and-inheritance)
- [Exercise 6 – Ordering and Hashing](#exercise-6--ordering-and-hashing)
- [Exercise 7 – `asdict` and `replace`](#exercise-7--asdict-and-replace)
- [Exercise 8 – Convert to/from dict](#exercise-8--convert-tofrom-dict)
- [Exercise 9 – Trading Case Study: Orders and Ticks](#exercise-9--trading-case-study-orders-and-ticks)


## Exercise 1 – Minimal Dataclass

**Goal**: Define a small dataclass and confirm `__init__`, `__repr__`, and `__eq__` behavior.

**Requirements**:

- Implement `Order(symbol: str, side: str, quantity: float)`.
- Two equal `Order` objects should compare as True.
- Print the dataclass instance (use default `repr`).

**Hint**: `@dataclass` generates boilerplate automatically.


In [ ]:
# Exercise 1 – implement Order dataclass

import numpy
from pandas import Timestamp
from dataclasses import dataclass, field

@dataclass
class Order:
    symbol: str; qty: float
    _MAX_QTY: float = 100
    _MIN_QTY: float = 0.01

    def __post_init__(self):
        sign = numpy.sign(self.qty)
        self.qty = abs(self.qty)
        self.symbol = self.symbol.upper()
        assert (self.qty >= self._MIN_QTY)
        self.qty = min(self.qty, self._MAX_QTY)
        self.qty = float(self.qty * sign)
        self.time = Timestamp.utcnow()
        t = int(self.time.timestamp() * 1e6)
        self.iid = numpy.base_repr(t, 36)

    def __init__(self, symbol: str, qty: float):
        self.symbol, self.qty = symbol, qty
        self.__post_init__()

    def __repr__(self):
        # Only include symbol and qty in the repr
        classname = type(self).__name__
        return f"{classname}({self.iid}, {self.qty!r} @ {self.symbol!r})"

    @property
    def MAX_QTY(self): return type(self)._MAX_QTY
    @property
    def MIN_QTY(self): return type(self)._MIN_QTY

args = ("EURUSD", -0.01)
o1, o2 = Order(*args), Order(*args)
o1, o2, o1 == o2

(Order(HH1JMKFAP6, -0.01 @ 'EURUSD'),
 Order(HH1JMKFAQ2, -0.01 @ 'EURUSD'),
 True)

## Exercise 2 – Defaults and `field()`

**Goal**: Use defaults and `default_factory` to avoid shared mutable state.

**Requirements**:

- Implement `Basket(items: list[str] = default_factory)`.
- Add a method `add(symbol: str)` that appends to the list.
- Demonstrate that two baskets don’t share the same list.

**Hint**: Prefer `field(default_factory=list)` for lists.


In [62]:
# Exercise 2 – implement Basket with safe defaults

from dataclasses import dataclass, field

@dataclass
class Basket:
    items: set = field(default_factory = set)
    def add(self, symbols: set[str]):
        self.items.update(symbols)

array = {"EURUSD", "AUDUSD"}
bk1, bk2 = Basket(), Basket()
bk1.add(array), bk2.add(array)
bk1.add({"GBPUSD"})
bk1, bk2, bk1 == bk2

(Basket(items={'AUDUSD', 'GBPUSD', 'EURUSD'}),
 Basket(items={'AUDUSD', 'EURUSD'}),
 False)

## Exercise 3 – `__post_init__` Validation

**Goal**: Validate inputs after dataclass initialization.

**Requirements**:

- Implement `ValidatedQty(quantity: float)`.
- In `__post_init__`, raise `ValueError` if `quantity <= 0`.
- Run the validation for a valid value and confirm it raises for an invalid value.

**Hint**: `__post_init__` runs after `__init__`.


In [67]:
# Exercise 3 – implement ValidatedQty

import numpy
from pandas import Timestamp
from dataclasses import dataclass, field

@dataclass
class Order:
    symbol: str; qty: float
    _MAX_QTY: float = 100
    _MIN_QTY: float = 0.01

    def __post_init__(self):
        sign = numpy.sign(self.qty)
        self.qty = abs(self.qty)
        self.symbol = self.symbol.upper()
        if (self.qty < self._MIN_QTY):
            err = f"\"{self.qty} < {self._MIN_QTY}\""
            raise ValueError(f"Quantity below min: {err}")
            
        self.qty = min(self.qty, self._MAX_QTY)
        self.qty = float(self.qty * sign)
        self.time = Timestamp.utcnow()
        t = int(self.time.timestamp() * 1e6)
        self.iid = numpy.base_repr(t, 36)

    def __init__(self, symbol: str, qty: float):
        self.symbol, self.qty = symbol, qty
        self.__post_init__()

    def __repr__(self):
        # Only include symbol and qty in the repr
        classname = type(self).__name__
        return f"{classname}({self.iid}, {self.qty!r} @ {self.symbol!r})"

    @property
    def MAX_QTY(self): return type(self)._MAX_QTY
    @property
    def MIN_QTY(self): return type(self)._MIN_QTY

try: Order("EURUSD", 0.00178)
except Exception as EXC: print(repr(EXC))

ValueError('Quantity below min: "0.00178 < 0.01"')


## Exercise 4 – `frozen=True` Immutability

**Goal**: Understand how `frozen=True` affects attribute assignment.

**Requirements**:

- Implement `FrozenOrder(symbol: str, quantity: int)` with `@dataclass(frozen=True)`.
- Attempt to modify `quantity` after creation; catch the resulting exception.
- In `__post_init__`, show how you *can* set/normalize fields using `object.__setattr__`.

**Hint**: Use `object.__setattr__(self, "quantity", new_value)`.


In [69]:
# Exercise 4 – implement FrozenOrder

from dataclasses import dataclass

@dataclass(frozen = True)
class FrozenOrder:
    symbol: str = field()
    qty: float = field()
    def __post_init__(self):
        side = "BUY" if (self.qty > 0) else "SELL"
        object.__setattr__(self, "side", side)
        object.__setattr__(self, "qty", abs(self.qty))
    def __repr__(self):
        cname = self.__class__.__name__
        return cname + f"({self.side} {self.qty} @ {self.symbol})"

o1 = FrozenOrder("EURUSD", +0.01)
o2 = FrozenOrder("GBPUSD", -0.01)
try: o2.qty = -0.02
except Exception as EXC:
    print(repr(EXC))
o1, o2

FrozenInstanceError("cannot assign to field 'qty'")


(FrozenOrder(BUY 0.01 @ EURUSD), FrozenOrder(SELL 0.01 @ GBPUSD))

## Exercise 5 – `slots=True` and Inheritance

**Goal**: Combine `slots=True` with dataclass inheritance.

**Requirements**:

- Implement `BaseTick(ts: int, symbol: str)` with `@dataclass(slots=True)`.
- Implement `TradeTick(BaseTick)` adding `price: float`.
- Instantiate `TradeTick` and show that it has both base and derived fields.

**Hint**: Inheritance works, but keep field names consistent.


In [ ]:
# Exercise 5 – implement BaseTick and TradeTick

from dataclasses import dataclass
from abc import ABC, abstractmethod
from pandas import Timestamp

@dataclass(slots = True)
class BaseTick(ABC):
    venue: str
    symbol: str
    def __post_init__(self):
        self.time = Timestamp.utcnow()
    @property
    @abstractmethod
    def str_data(self): ...
    def __repr__(self):
        time = self.time.isoformat(" ", "microseconds")[: 26]
        return f"Tick({time} :: {self.venue}|{self.symbol} :: {self.str_data})"

@dataclass(repr = False)
class TradeTick(BaseTick):
    pa: float; pb: float
    va: float; vb: float
    @property
    def str_data(self):
        return f"A{self.va:.2f}@{self.pa:.5f} B{self.vb:.2f}@{self.pb:.5f}"

TradeTick("IBKR", "EURUSD", 1.23456, 1.23455, 100, 50)
# TradeTick(2026-03-27 08:46:43.100322 :: IBKR|EURUSD :: A100@1.23456 B50@1.23455)

Tick(2026-03-27 08:48:14.352667 :: IBKR|EURUSD :: A100.00@1.23456 B50.00@1.23455)

## Exercise 6 – Ordering and Hashing

**Goal**: Use `order=True` and understand how hashing works.

**Requirements**:

- Implement `PriceLevel(price: float, size: float)` as `@dataclass(order=True)`.
- Compare two levels using `<`.
- Attempt to use a `PriceLevel` in a `set`. If it fails, explain why (hashing rules) and adjust by either using `frozen=True` or `unsafe_hash=True`.

**Hint**: `order=True` and hashability are related to immutability.


In [92]:
# Exercise 6 – implement PriceLevel ordering + hashing experiment

from dataclasses import dataclass

@dataclass(order = True)
class PriceLevel:
    price: float
    size: float

pl1 = PriceLevel(123.456, 123)
pl2 = PriceLevel(123.457, 456)
print(f"\"{pl1 = } < {pl2 = }\" ==> {pl1 < pl2}")

try: pls = { # should return error
    PriceLevel(1.23456, 100),
    PriceLevel(1.23456, 100),
    PriceLevel(1.23456, 100),
    PriceLevel(1.23457, 90),
    PriceLevel(1.23457, 80),
    PriceLevel(1.23458, 40), }
except Exception as EXC:
    print(repr(EXC))

@dataclass(order = True, frozen = True)
class FrozenPriceLevel:
    price: float
    size: float

try: pls = { # shouldn't return error
    FrozenPriceLevel(1.23456, 100),
    FrozenPriceLevel(1.23456, 100),
    FrozenPriceLevel(1.23456, 100),
    FrozenPriceLevel(1.23457, 90),
    FrozenPriceLevel(1.23457, 80),
    FrozenPriceLevel(1.23458, 40), }
except Exception as EXC:
    print(repr(EXC))

"pl1 = PriceLevel(price=123.456, size=123) < pl2 = PriceLevel(price=123.457, size=456)" ==> True
TypeError("unhashable type: 'PriceLevel'")


## Exercise 7 – `asdict` and `replace`

**Goal**: Convert to a dict and create modified copies.

**Requirements**:

- Implement `RiskConfig(max_loss: float, max_leverage: float = 1.0)`.
- Use `asdict()` to convert it to a dict.
- Use `replace(config, max_loss=...)` to create a modified copy.

**Hint**: `replace` returns a new instance.


In [96]:
# Exercise 7 – implement RiskConfig and test asdict/replace

from dataclasses import dataclass, field, asdict, replace

@dataclass
class RiskConfig:
    max_loss: float = field(default = 1.0)
    max_leverage: int = field(default = 1000)

rc1 = RiskConfig(0.1, 10) ; rc2 = replace(rc1, max_loss = 0.01)
print(f"Original instance ==> {rc1 = }\nAs dict ==> {asdict(rc1)}")
print(f"Modified instance ==> {rc2 = }\nAs dict ==> {asdict(rc2)}")

Original instance ==> rc1 = RiskConfig(max_loss=0.1, max_leverage=10)
As dict ==> {'max_loss': 0.1, 'max_leverage': 10}
Modified instance ==> rc2 = RiskConfig(max_loss=0.01, max_leverage=10)
As dict ==> {'max_loss': 0.01, 'max_leverage': 10}


## Exercise 8 – Convert to/from dict

**Goal**: Provide explicit serialization helpers.

**Requirements**:

- Implement `class Tick` dataclass with `to_dict()` and `from_dict()`.
- `from_dict` should validate required keys and types (at least basic checks).
- Demo: round-trip `Tick -> dict -> Tick`.

**Hint**: Keep validation simple; the focus is the pattern.


In [106]:
# Exercise 8 – implement Tick serialization

from dataclasses import dataclass, asdict
from pandas import Timestamp
from typing import Dict

@dataclass
class Tick:
    venue: str; symbol: str
    pa: float; pb: float
    va: float; vb: float
    def __post_init__(self):
        self.time = Timestamp.utcnow()
    @classmethod
    def from_dict(cls, tick: Dict):
        assert (tick["pa"] > 0) and (tick["pb"] > 0)
        assert (tick["va"] > 0) and (tick["vb"] > 0)
        return cls(**tick)
    def to_dict(self):
        return {"time": self.time, **asdict(self)}
    @property
    def str_data(self):
        return f"A{self.va:.2f}@{self.pa:.5f} B{self.vb:.2f}@{self.pb:.5f}"
    def __repr__(self):
        time = self.time.isoformat(" ", "microseconds")[: 26]
        return f"Tick({time} :: {self.venue}|{self.symbol} :: {self.str_data})"

tick_1 = Tick("IBKR", "EURUSD", 1.23457, 1.23456, 100, 50)
dict_1 = tick_1.to_dict()
dict_1.pop("time")
tick_2 = Tick.from_dict(dict_1)
tick_1, tick_2, tick_1 == tick_2

(Tick(2026-03-27 09:06:31.001542 :: IBKR|EURUSD :: A100.00@1.23457 B50.00@1.23456),
 Tick(2026-03-27 09:06:31.001604 :: IBKR|EURUSD :: A100.00@1.23457 B50.00@1.23456),
 True)

## Exercise 9 – Trading Case Study: Orders and Ticks

**Goal**: Use multiple dataclasses together and add a small piece of domain logic.

**Requirements**:

- Implement `Order(symbol: str, side: str, qty: float)` and `Tick(ts: int, symbol: str, bid: float, ask: float)`.
- Add a function `mid_price(tick)` returning `(bid + ask) / 2`.
- Add a function `estimated_execution_price(order, tick)`:
- for `BUY` return `tick.ask`
- for `SELL` return `tick.bid`
- Demo with one order and one tick.

**Hint**: Use methods or plain functions—either is fine.


In [ ]:
# Exercise 9 – trading case study

from dataclasses import dataclass